## Stockholms stads offentliga toaletter
* [#1](https://github.com/salgo60/Offentliga-toaletter/issues/1) / [#19](https://github.com/salgo60/Offentliga-toaletter/issues/19)
   * karta [Toaletter_sthlm_latest.html](https://salgo60.github.io/Offentliga-toaletter/Notebook/Toaletter_sthlm_latest.html)
   * karta [Toaletter_sthml_0_9.html](https://salgo60.github.io/Offentliga-toaletter/Notebook/Toaletter_sthml_0_9.html)
* Notebook [Toaletter_1_1_Sthlm.ipynb](https://github.com/salgo60/Offentliga-toaletter/blob/main/Notebook/Toaletter_1_1_Sthlm.ipynb) 

---- 
* 0.1 skapade
* 0.2 la till länkar Googfle Map etc... i popup
* 0.3 söker fram toaletter i OSM i dag
* 0.4 visar OSM egenskaper och bild wikicommons
* 0.5 visar även OSM disused:amenity toilets
* 0.6 skapar ett jcdecaux_layer
* 0.7 försökt matcha appen Toa Sverige med OSM data se
* 0.8 Added Danfo layer
* 0.9 Nytt dataset Toalett_Punkt_Sthklm_20260505 [#19](https://github.com/salgo60/Offentliga-toaletter/issues/19)


In [1]:
import time
from datetime import datetime 
import glob
import pandas as pd
from datetime import date

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))



Start: 2026-05-05 11:49:10


In [2]:
# Dopwnload gpkgpacket before... 
#
# https://openstreetgs.stockholm.se/geoservice/api/ba9e5991-379f-4eb4-b6a3-e288a3730b2a/wfs/?version=1.0.0&request=GetFeature&typeName=od_gis:Toalett_Punkt&outputFormat=GeoPackage
#
SthlmfileName =f"data/raw/Toalett_Punkt_Sthlm_20260123.gpkg"
SthlmfileName =f"data/raw/Toalett_Punkt_Sthlm_20260210.gpkg"
SthlmfileName =f"data/raw/Toalett_Punkt_Sthlm_20260505.gpkg"
Mapversion = "0.9"

# used by part 2
#output_path = f"data/raw/Toalett_Punkt_Sthlm_{today}.gpkg"
output_path = f"data/raw/Toalett_Punkt_Sthlm_20260210.gpkg"
output_path = f"data/raw/Toalett_Punkt_Sthlm_20260505.gpkg"


In [3]:
import geopandas as gpd

gdf_raw = gpd.read_file(output_path)

print("Antal objekt:", len(gdf_raw))
print("Kolumner:", list(gdf_raw.columns))
gdf_raw.head(3)


Antal objekt: 91
Kolumner: ['id', 'Placering', 'Huvudman', 'Drift', 'Typ', 'Modell', 'Stadsdelsnämnder', 'Anläggningsnummer', 'Cykelpump', 'Dricksfontän', 'geometry']


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'data/raw/Toalett_Punkt_Sthlm_20260505.gpkg'
  return ogr_read(


,id,Placering,Huvudman,Drift,Typ,Modell,Stadsdelsnämnder,Anläggningsnummer,Cykelpump,Dricksfontän,geometry
0,102,Axel Landquist Park - Åsögatan 146,TK,Danfo,Urinoar,Urinoar,Södermalm,G101,Nej,Nej,POINT (154608.759 6577634.818)
1,104,Strömparterren 3,TK,Danfo,Toalett,Övrigt,Södermalm,G079,Nej,Nej,POINT (153988.84 6579271.263)
2,105,Fjällgatan Södermalm,TK,Miljötekniska,Toalett,Enkel,Södermalm,G085,Nej,Nej,POINT (154957.149 6578024.61)


In [4]:
from pathlib import Path


def popup_with_history(row, df_history):
    return folium.Popup(
        f"""
        <b>Namn:</b> {row.name_old}<br>
        <b>Ändring:</b> Attribut ändrade<br><br>
        <b>Historik:</b><br>
        {summarize_history(df_history, row.stable_id)}<br><br>
        <a href="https://www.openstreetmap.org/?mlat={row.lat_new}&mlon={row.lon_new}#map=19/{row.lat_new}/{row.lon_new}" target="_blank">
            Visa i OSM
        </a>
        """,
        max_width=400
    )


In [5]:
import glob
import pandas as pd
from pathlib import Path

def build_history(processed_dir="data/processed"):
    dfs = []

    for path in sorted(glob.glob(f"{processed_dir}/stockholm_*.parquet")):
        date = Path(path).stem.split("_")[-1]
        df = pd.read_parquet(path)

        df = df.assign(date=pd.to_datetime(date))
        dfs.append(df[[
            "stable_id",
            "date",
            "lat",
            "lon",
            "opening_hours",
            "accessibility",
            "name"
        ]])

    return pd.concat(dfs, ignore_index=True)

In [6]:
df_history = build_history()
print("Historikposter:", len(df_history))


Historikposter: 1101


In [7]:
def summarize_history(df_hist, stable_id):
    h = df_hist[df_hist.stable_id == stable_id].sort_values("date")

    if h.empty:
        return "Ingen historik"

    first = h.iloc[0]["date"]
    last = h.iloc[-1]["date"]
    count = len(h)

    moved = (
        h[["lat", "lon"]]
        .diff()
        .abs()
        .sum(axis=1)
        .fillna(0)
        .gt(0.00001)
        .any()
    )

    status_changes = (
        h["opening_hours"].nunique() > 1 or
        h["accessibility"].nunique() > 1
    )

    lines = [
        f"Första observation: {first}",
        f"Senaste observation: {last}",
        f"Antal versioner: {count}",
        f"Flyttad över tid: {'Ja' if moved else 'Nej'}",
        f"Status ändrad: {'Ja' if status_changes else 'Nej'}",
    ]

    return "<br>".join(lines)


In [8]:
import folium 
mdiff = folium.Map(
    location=[59.3293, 18.0686],
    zoom_start=11,
    tiles="OpenStreetMap"
)


In [9]:
def popup(d):
    return folium.Popup(
        "<br>".join(f"<b>{k}</b>: {v}" for k, v in d.items()),
        max_width=350
    )


In [10]:
def timeline_table(df_hist, stable_id, limit=5):
    h = df_hist[df_hist.stable_id == stable_id].sort_values("date")
    if h.empty:
        return "Ingen historik"

    rows = h.tail(limit)

    html = "<table border='1' style='border-collapse:collapse;font-size:12px;'>"
    html += "<tr><th>Datum</th><th>Öppettider</th><th>Tillg.</th></tr>"

    for _, r in rows.iterrows():
        html += (
            f"<tr>"
            f"<td>{r.date}</td>"
            f"<td>{r.opening_hours}</td>"
            f"<td>{r.accessibility}</td>"
            f"</tr>"
        )

    html += "</table>"
    if len(h) > limit:
        html += f"<i>Visar {limit} av {len(h)} versioner</i>"

    return html


In [11]:
from datetime import datetime

def inactive_days(df_hist, stable_id):
    h = df_hist[df_hist.stable_id == stable_id].sort_values("date")
    if len(h) < 2:
        return 0

    last_change = h[h["opening_hours"].ne(h["opening_hours"].shift()) |
                    h["accessibility"].ne(h["accessibility"].shift())]

    if last_change.empty:
        first = datetime.strptime(h.iloc[0]["date"], "%Y%m%d")
    else:
        first = datetime.strptime(last_change.iloc[-1]["date"], "%Y%m%d")

    last = datetime.strptime(h.iloc[-1]["date"], "%Y%m%d")
    return (last - first).days


In [12]:
def stockholm_toalett_adapter(gdf):
    df = gdf.copy()

    # säkerställ punktgeometri
    df = df[df.geometry.notnull()]
    df["lon"] = df.geometry.x
    df["lat"] = df.geometry.y

    out = pd.DataFrame({
        "name": df.get("NAMN", df.get("namn")),
        "lat": df["lat"],
        "lon": df["lon"],
        "opening_hours": df.get("OPPETTIDER", df.get("oppettider")),
        "accessibility": df.get("TILLGANG", df.get("tillganglighet")),
    })

    # rensa minimalt
    out["name"] = out["name"].astype(str).str.strip()

    # stabilt ID (≈ 1 m)
    out["stable_id"] = (
        out["lat"].round(5).astype(str)
        + "_"
        + out["lon"].round(5).astype(str)
    )

    return out.dropna(subset=["lat", "lon"])


In [13]:
df_new = stockholm_toalett_adapter(gdf_raw)
print("Adapter-resultat:", len(df_new))

Adapter-resultat: 91


In [14]:
df_old = None
added = pd.DataFrame()
removed = pd.DataFrame()
changed_geom = pd.DataFrame()
changed_attrs = pd.DataFrame()

if df_old is not None:
    added = df_new[~df_new["stable_id"].isin(df_old["stable_id"])]
    removed = df_old[~df_old["stable_id"].isin(df_new["stable_id"])]

    merged = df_old.merge(
        df_new,
        on="stable_id",
        suffixes=("_old", "_new")
    )

    changed_geom = merged[
        (merged["lat_old"] != merged["lat_new"]) |
        (merged["lon_old"] != merged["lon_new"])
    ]

    changed_attrs = merged[
        (merged["opening_hours_old"] != merged["opening_hours_new"]) |
        (merged["accessibility_old"] != merged["accessibility_new"])
    ]


In [15]:

from datetime import date

In [16]:
import folium 
fg_added = folium.FeatureGroup(name="🟢 Tillkomna")
today = date.today().strftime("%Y%m%d") 



for _, r in added.iterrows():
    popup_html = f"""
        <b>Namn:</b> {r.name_old}<br>
        <b>Ändring:</b> Attribut ändrade<br><br>
        
        <b>Historik:</b><br>
        {timeline_table(df_history, r.stable_id)}<br><br>
        
        <a href="https://www.openstreetmap.org/?mlat={r.lat_new}&mlon={r.lon_new}#map=19/{r.lat_new}/{r.lon_new}" target="_blank">
        Visa i OSM
        </a>
        """
    folium.CircleMarker(
        [r.lat, r.lon],
        radius=6,
        color="green",
        fill=True,
        fill_opacity=0.8,
        popup=popup({
            "Namn": r.name,
            "Ändring": "Tillkommen"
        })
    ).add_to(fg_added)

fg_added.add_to(mdiff) 

fg_removed = folium.FeatureGroup(name="🔴 Borttagna")

for _, r in removed.iterrows():
    folium.CircleMarker(
        [r.lat, r.lon],
        radius=6,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=popup({
            "Namn": r.name,
            "Ändring": "Borttagen"
        })
    ).add_to(fg_removed)

fg_removed.add_to(mdiff)

fg_moved = folium.FeatureGroup(name="🟠 Flyttade")

for _, r in changed_geom.iterrows():
    old = (r.lat_old, r.lon_old)
    new = (r.lat_new, r.lon_new)

    folium.PolyLine([old, new], color="orange", weight=2).add_to(fg_moved)

    folium.CircleMarker(old, radius=5, color="orange", fill=True).add_to(fg_moved)
    folium.CircleMarker(
        new, radius=7, color="orange", fill=True,
        popup=popup({
            "Namn": r.name_old,
            "Gammal position": old,
            "Ny position": new
        })
    ).add_to(fg_moved)

fg_moved.add_to(mdiff)

fg_changed = folium.FeatureGroup(name="🟡 Ändrade attribut")

for _, r in changed_attrs.iterrows():
    folium.CircleMarker(
        [r.lat_new, r.lon_new],
        radius=6,
        color="gold",
        fill=True,
        fill_opacity=0.8,
        popup=popup_with_history(r, df_history)
    ).add_to(fg_changed)


fg_changed.add_to(mdiff)

folium.LayerControl(collapsed=False).add_to(mdiff)

legend_html = f"""
<div style="
 position: fixed;
 bottom: 30px;
 left: 30px;
 z-index: 9999;
 background-color: white;
 padding: 10px;
 border: 2px solid grey;
 font-size: 14px;
">
<b>Legend – Toaletter Stockholm (diff)</b><br>
<span style="color:green;">●</span> Tillkommen<br>
<span style="color:red;">●</span> Borttagen<br>
<span style="color:orange;">●</span> Flyttad<br>
<span style="color:gold;">●</span> Attribut ändrat<br>
<hr> Analys: versionsjämförelse {today}<br /><br />
<b> Mer info</b><br>
• <a href="https://github.com/salgo60/Offentliga-toaletter/issues/19" target="_blank">GitHub Issue #19</a><br>
• <a href="https://dataportalen.stockholm.se/dataportalen/GetMetaDataById?id=LvFeature15303196&showmetadataview" target="_blank">
   Stockholms stads öppna data</a>
</div>
"""
mdiff.get_root().html.add_child(folium.Element(legend_html))

mdiff_Filename = f"Toaletter_sthlm_diff_{today}.html"
mdiff.save(mdiff_Filename)

print("Klar: ", mdiff_Filename)


Klar:  Toaletter_sthlm_diff_20260505.html


In [17]:
mdiff

In [18]:
from pathlib import Path 
Path("data/processed").mkdir(parents=True, exist_ok=True)

processed_files = sorted(glob.glob("data/processed/stockholm_*.parquet"))
df_old = pd.read_parquet(processed_files[-1]) if processed_files else None

df_new.to_parquet(f"data/processed/stockholm_{today}.parquet")

added = removed = changed_geom = changed_attrs = pd.DataFrame()

if df_old is not None:
    added = df_new[~df_new.stable_id.isin(df_old.stable_id)]
    removed = df_old[~df_old.stable_id.isin(df_new.stable_id)]

    merged = df_old.merge(
        df_new,
        on="stable_id",
        suffixes=("_old", "_new")
    )

    changed_geom = merged[
        (merged.lat_old != merged.lat_new) |
        (merged.lon_old != merged.lon_new)
    ]

    changed_attrs = merged[
        (merged.opening_hours_old != merged.opening_hours_new) |
        (merged.accessibility_old != merged.accessibility_new)
    ]

print(
    "Tillkomna:", len(added),
    "Borttagna:", len(removed),
    "Flyttade:", len(changed_geom),
    "Ändrade attribut:", len(changed_attrs)
)


Tillkomna: 0 Borttagna: 0 Flyttade: 0 Ändrade attribut: 91


## Karta Offentliga toaletter 
* visat karta med OSM offentliga toaletter och Stockholms stad

In [19]:
import geopandas as gpd

# Läs in geopackage-filen
gdf = gpd.read_file(SthlmfileName)

# Visa grundläggande information
print(gdf.head())
print(gdf.crs)
print(gdf.columns)

    id                             Placering Huvudman          Drift      Typ  \
0  102    Axel Landquist Park - Åsögatan 146       TK          Danfo  Urinoar   
1  104                      Strömparterren 3       TK          Danfo  Toalett   
2  105                Fjällgatan   Södermalm       TK  Miljötekniska  Toalett   
3  106  Folkungagatan 82 - Parkleken Droskan       TK  Miljötekniska  Urinoar   
4  107                      Frödingsvägen 12       TK  Miljötekniska  Toalett   

    Modell Stadsdelsnämnder Anläggningsnummer Cykelpump Dricksfontän  \
0  Urinoar        Södermalm              G101       Nej          Nej   
1   Övrigt        Södermalm              G079       Nej          Nej   
2    Enkel        Södermalm              G085       Nej          Nej   
3  Urinoar        Södermalm              G105       Nej          Nej   
4    Enkel      Kungsholmen              G084       Nej          Nej   

                         geometry  
0  POINT (154608.759 6577634.818)  
1   POIN

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'data/raw/Toalett_Punkt_Sthlm_20260505.gpkg'
  return ogr_read(


In [20]:
def convert_misread_date(x):
    try:
        if pd.isna(x):
            return None
        # gör till int, ta bara siffrorna
        s = str(int(float(x)))
        # om 8 tecken → YYYYMMDD
        if len(s) == 8:
            return pd.to_datetime(s, format="%Y%m%d", errors="coerce")
        return None
    except Exception:
        return None

for col in [c for c in gdf.columns if "DATE" in c.upper()]:
    gdf[col + "_fmt"] = gdf[col].apply(convert_misread_date)

In [21]:
import geopandas as gpd
import pandas as pd

# Läs in
gdf = gpd.read_file(SthlmfileName)

# Beräkna centroid i meter-CRS (SWEREF99 TM) och konvertera till WGS84
gdf_proj = gdf.to_crs(epsg=3006)
gdf["geometry_centroid"] = gdf_proj.geometry.centroid.to_crs(epsg=4326)

# Lägg till lat/lon
gdf["lon"] = gdf["geometry_centroid"].x
gdf["lat"] = gdf["geometry_centroid"].y

# Lista möjliga datumkolumner
date_cols = [c for c in gdf.columns if "DATE" in c.upper() or "DATUM" in c.upper()]
print("🕒 Datumkolumner:", date_cols)

# Funktion för att försöka tolka stora siffror som tidsstämplar
def convert_date(x):
    try:
        # Om NaN
        if pd.isna(x):
            return None
        # Om redan datetime
        if isinstance(x, pd.Timestamp):
            return x
        # Om det ser ut som 1.738443e+15 (millisekunder)
        x = float(x)
        if x > 1e12:  # troligen millisekunder
            return pd.to_datetime(x, unit="ms")
        elif x > 1e9:  # sekunder
            return pd.to_datetime(x, unit="s")
        else:
            return None
    except Exception:
        return None

# Konvertera alla identifierade datumfält
for col in date_cols:
    gdf[col + "_fmt"] = gdf[col].apply(convert_date)

# Visa exempel på konverterade datum
print(gdf[[c for c in gdf.columns if "DATE" in c or "fmt" in c]].head())

gdf.drop(columns=["geometry", "geometry_centroid"]).to_csv("SthlmfileName_Toalett_fix.csv", index=False)
print("✅ Sparade 'SthlmfileName_Toalett_fix.csv' med riktiga datum och koordinater")

🕒 Datumkolumner: []
Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]
✅ Sparade 'SthlmfileName_Toalett_fix.csv' med riktiga datum och koordinater


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'data/raw/Toalett_Punkt_Sthlm_20260505.gpkg'
  return ogr_read(


In [22]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   id                 91 non-null     int64   
 1   Placering          91 non-null     object  
 2   Huvudman           91 non-null     object  
 3   Drift              91 non-null     object  
 4   Typ                91 non-null     object  
 5   Modell             91 non-null     object  
 6   Stadsdelsnämnder   91 non-null     object  
 7   Anläggningsnummer  91 non-null     object  
 8   Cykelpump          91 non-null     object  
 9   Dricksfontän       91 non-null     object  
 10  geometry           91 non-null     geometry
 11  geometry_centroid  91 non-null     geometry
 12  lon                91 non-null     float64 
 13  lat                91 non-null     float64 
dtypes: float64(2), geometry(2), int64(1), object(9)
memory usage: 10.1+ KB


In [23]:
gdf.head()

,id,Placering,Huvudman,Drift,Typ,Modell,Stadsdelsnämnder,Anläggningsnummer,Cykelpump,Dricksfontän,geometry,geometry_centroid,lon,lat
0,102,Axel Landquist Park - Åsögatan 146,TK,Danfo,Urinoar,Urinoar,Södermalm,G101,Nej,Nej,POINT (154608.759 6577634.818),POINT (18.08092 59.31386),18.080924,59.313856
1,104,Strömparterren 3,TK,Danfo,Toalett,Övrigt,Södermalm,G079,Nej,Nej,POINT (153988.84 6579271.263),POINT (18.07007 59.32855),18.070070,59.328552
2,105,Fjällgatan Södermalm,TK,Miljötekniska,Toalett,Enkel,Södermalm,G085,Nej,Nej,POINT (154957.149 6578024.61),POINT (18.08705 59.31735),18.087051,59.317352
3,106,Folkungagatan 82 - Parkleken Droskan,TK,Miljötekniska,Urinoar,Urinoar,Södermalm,G105,Nej,Nej,POINT (154510.3 6577769.004),POINT (18.0792 59.31506),18.079198,59.315062
4,107,Frödingsvägen 12,TK,Miljötekniska,Toalett,Enkel,Kungsholmen,G084,Nej,Nej,POINT (150164.167 6579432.37),POINT (18.00288 59.33002),18.002884,59.330017


In [24]:
cols = [
    "Placering",
    "Huvudman",
    "Drift",
    "Typ",
    "Modell",
    "Stadsdelsnämnder",
    "Anläggningsnummer",
    "Cykelpump",
    "Dricksfontän"
]

for col in cols:
    print(f"\n--- {col} ---")
    print(gdf[col].unique())


--- Placering ---
['Axel Landquist Park - Åsögatan 146' 'Strömparterren 3'
 'Fjällgatan   Södermalm' 'Folkungagatan 82 - Parkleken Droskan'
 'Frödingsvägen 12' 'Klara Mälarstrand 4 - Stadshuskajen'
 'Kronobergsgatan 11 - Kronobergsparken' 'Kroppaplan' 'Källargränd 4'
 'Liljeholmsgränd 17' 'Mikrofonvägen\n4' 'Nioörtsvägen 24'
 'Palmfeltsvägen 28 ' 'Skidvägen 11' 'Sockenplan 370'
 'Trollesundsvägen 16 (under tåg bron)'
 'Valhallavägen/ Drottning Sofias väg - i allén'
 'Olof Palmes Gata 31 - Norra bantorget'
 'Västertorpsvägen 70 \n(utanför T-bannan)' 'Ellen Keys gata 46'
 'Skarpskyttestigen 6 - Tantobadet' 'Skånegatan 70' 'Djurgårdsvägen 60'
 'Karlavägen 40' 'Söder Mälarstrand 115 - Under Västerbron'
 'Rågsved T-station' 'Hamngatan 2 - Nybroplan'
 'Akallastigen 6 - Akalla By' 'Husbystigen 9 - Husby Gård'
 'Mälartorget 19' 'Slottsbacken/Österlånggatan'
 'Österlånggatan 45 - Benickebrinken' 'Medborgarplatsen 54'
 'Skånegatan 79 - Nytorget' 'Allmänna Gränd - Djurgården'
 'Elersvägen 50 - K

In [25]:
for col in cols:
    print(f"\n=== {col} ===")
    print(gdf[col].value_counts())


=== Placering ===
Placering
Axel Landquist Park - Åsögatan 146      1
Odengatan 77 - Odenplan                 1
Norra Bantorget                         1
Norr Mälarstrand 3                      1
Nordenflychtsvägen 11 - Kristineberg    1
                                       ..
Husbystigen 9 - Husby Gård              1
Akallastigen 6 - Akalla By              1
Hamngatan 2 - Nybroplan                 1
Rågsved T-station                       1
Sveavägen 39A - Adolf Fredriks Kyrka    1
Name: count, Length: 91, dtype: int64

=== Huvudman ===
Huvudman
Danfo    65
TK       26
Name: count, dtype: int64

=== Drift ===
Drift
Danfo            68
Miljötekniska    23
Name: count, dtype: int64

=== Typ ===
Typ
Toalett    70
Urinoar    21
Name: count, dtype: int64

=== Modell ===
Modell
Enkel      41
Urinoar    21
Trippel    11
Dubbel      9
null        5
Övrigt      2
Stor        2
Name: count, dtype: int64

=== Stadsdelsnämnder ===
Stadsdelsnämnder
Södermalm               23
Norra innerstaden  

In [26]:
gdf.describe()

,id,lon,lat
count,91.000000,91.000000,91.000000
mean,153.109890,18.028740,59.324162
std,30.034221,0.063118,0.032652
min,102.000000,17.869377,59.243972
25%,125.500000,18.003163,59.307328
50%,153.000000,18.052176,59.326010
75%,179.500000,18.073664,59.339463
max,202.000000,18.130993,59.413187


In [27]:
#gdf["Antal_WC"].value_counts() 

In [28]:
#gdf["Antal_urinoarer"].value_counts() 

In [29]:
#gdf["Beskrivning"].value_counts()

### Check if Stockholms stads data has been updated

In [30]:
#gdf["CREATE_DATE"].value_counts() 
#gdf["CREATE_DATE"].astype("Int64").astype(str).str.len().value_counts()


In [31]:
#gdf["CREATE_DATE_DT"] = pd.to_datetime(
#    gdf["CREATE_DATE"].astype("Int64").astype(str),
#    format="%Y%m%d%H%M",
#    errors="coerce"
#)

In [32]:
#gdf["CHANGE_DATE_DT"] = pd.to_datetime(
#    gdf["CHANGE_DATE"].astype("Int64").astype(str),
#    format="%Y%m%d%H%M",
#    errors="coerce"
#)

In [33]:
#gdf[["CREATE_DATE", "CREATE_DATE_DT"]].sample(10)

In [34]:
#gdf_sorted_create = gdf.sort_values(by="CREATE_DATE_DT", ascending=False) 
#gdf_sorted_create[["CREATE_DATE", "CREATE_DATE_DT"]]

In [35]:
#gdf_sorted_change = gdf.sort_values(by="CHANGE_DATE_DT", ascending=False) 
#gdf_sorted_change[["CHANGE_DATE", "CHANGE_DATE_DT"]]

In [36]:
#gdf_sorted_change["CHANGE_DATE_DAY"] = gdf_sorted_change["CHANGE_DATE_DT"].dt.date

#gdf_sorted_change.groupby("CHANGE_DATE_DAY").size().reset_index(name="antal")

In [37]:
#gdf[["CHANGE_DATE", "CHANGE_DATE_DT"]].sample(10)

In [38]:
#gdf["Driftansvar"].value_counts() 
gdf["Huvudman"].value_counts() 

Huvudman
Danfo    65
TK       26
Name: count, dtype: int64

In [39]:
from shapely.geometry import base

for col in gdf.columns:
    has_geom = gdf[col].apply(lambda x: isinstance(x, base.BaseGeometry)).any()
    if has_geom:
        print(f"🧱 Kolumn '{col}' innehåller geometrier")


🧱 Kolumn 'geometry' innehåller geometrier
🧱 Kolumn 'geometry_centroid' innehåller geometrier


In [40]:
from urllib.parse import unquote

def build_commons_thumbnail(tags: dict, width: int = 240) -> str:
    image_url = tags.get("image")
    if not image_url:
        return ""

    if "/wiki/File:" not in image_url:
        return ""

    filename = image_url.split("/wiki/File:")[-1]
    filename = unquote(filename)  # hantera %C3%A5 etc korrekt

    thumb_url = (
        "https://commons.wikimedia.org/wiki/Special:FilePath/"
        f"{filename}?width={width}"
    )

    return f"""
    <div style="margin-top:6px;text-align:center;">
      <a href="{image_url}" target="_blank">
        <img src="{thumb_url}"
             style="max-width:100%;
                    border-radius:6px;
                    box-shadow:0 1px 4px rgba(0,0,0,0.3);"
             loading="lazy"
             alt="Wikimedia Commons image">
      </a>
      <div style="font-size:12px;color:#555;margin-top:2px;">
        Klicka för full storlek (Wikimedia Commons)
      </div>
    </div>
    """


In [41]:
def build_toilet_tooltip(row):
    operator = row.get("Driftansvar", "")
    description = row.get("Beskrivning", "")
    change_dt = row.get("CHANGE_DATE_DT", "")
    create_dt = row.get("CREATE_DATE_DT", "")

    avtal_html = ""
    if operator == "JCDecaux":
        avtal_html = "<b style='color:#7b2cff;'>⚠ Avtal uppsagt (JCDecaux)</b><br>"
    if operator == "JCDecaux":
        avtal_html = "<b style='color:#7b2cff;'>⚠ Avtal uppsagt (JCDecaux)</b><br>"

    tooltip_html = f"""
    <div style="font-size:12px;">
      {avtal_html}
      <b>Beskrivning:</b> {description}<br>
      <b>Skapad:</b> {create_dt}<br>
      <b>Ändrad:</b> {change_dt}
    </div>
    """

    return tooltip_html


In [42]:
from urllib.parse import quote

def build_toilet_popup(row, *, include_source=True):    
    lat = row["lat"]
    lon = row["lon"]

    name = "Offentlig toalett"
    address = row.get("Adress", "")
    status = row.get("Status", "")
    operator = row.get("Driftansvar", "Stockholms stad")
    accessible = row.get("Tillgänglighetsanpassad", "")
    facility_id = row.get("Anläggningsnummer", "")
    operator = row.get("Driftansvar", "")
    description = row.get("Beskrivning", "")
    change_dt = row.get("CHANGE_DATE_DT", "")
    create_dt = row.get("CREATE_DATE_DT", "")

    avtal_html = ""
    if operator == "JCDecaux":
        avtal_html = "<b style='color:#7b2cff;'>⚠ Avtal uppsagt (JCDecaux)</b><br>"


    # --- Tillgänglighet ---
    wheelchair_html = {
        "Ja": "♿ <b>Tillgänglig:</b> Ja ✅<br>",
        "Nej": "♿ <b>Tillgänglig:</b> Nej ❌<br>",
    }.get(accessible, "♿ <b>Tillgänglig:</b> Okänd ❔<br>")

    # --- OSM Note ---
    note_text = (
        f"Kommunal toalett enligt Stockholms stad. "
        f"Adress: {address}. "
        f"Tillgänglig: {accessible}. "
        f"Anläggningsnummer: {facility_id}."
    )
    note_url = (
        f"https://www.openstreetmap.org/note/new"
        f"#map=18/{lat}/{lon}&text={quote(note_text)}"
    )

    mapcomplete = f"https://mapcomplete.org/toilets.html?z=17&lat={lat}&lon={lon}#about_theme"
    wheelmap = f"https://wheelmap.org/map#/?lat={lat}&lon={lon}&zoom=17"
    # --- App / editor-länkar ---
    google_maps = f"https://maps.google.com/?q={lat},{lon}"
    streetview = f"https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={lat},{lon}"
    wikishootme = f"https://wikishootme.toolforge.org/#lat={lat}&lng={lon}&zoom=17"
    
    menu_html = f"""
        <div style='margin-top:8px; text-align:center;'>
          <a href='{mapcomplete}' target='_blank'
             style='background:#34a853;color:white;padding:6px 10px;
                    border-radius:6px;text-decoration:none;margin:2px;
                    display:inline-block;font-size:13px;'>
             🚻 MapComplete – Toilets
          </a>
        
          <a href='{wheelmap}' target='_blank'
             style='background:#7c3aed;color:white;padding:6px 10px;
                    border-radius:6px;text-decoration:none;margin:2px;
                    display:inline-block;font-size:13px;'>
             ♿ Wheelmap
          </a>
        
          <a href='{note_url}' target='_blank'
             style='background:#fbbc04;color:black;padding:6px 10px;
                    border-radius:6px;text-decoration:none;margin:2px;
                    display:inline-block;font-size:13px;'>
             📝 OSM Note
          </a>
        </div>
        """

    tools_html = f"""
    <details style='margin-top:6px; text-align:center;'>
      <summary style='cursor:pointer;font-weight:bold;'>🔗 Fler kartverktyg</summary>
      <a href='https://www.openstreetmap.org/edit#map=18/{lat}/{lon}' target='_blank'>✏️ iD Editor</a><br>
      <a href='https://www.mapillary.com/app/?lat={lat}&lng={lon}&z=17' target='_blank'>🟢 Mapillary</a><br>
      <a href='{google_maps}' target='_blank'>🗺️ Google Maps</a><br>
      <a href='{streetview}' target='_blank'>👁️ Google Street View</a><br>
      <a href='{wikishootme}' target='_blank'>🧭 Wikishootme</a><br>
    </details>
    """

    source_html = ""
    if include_source:
        source_html ="""<hr style="margin:6px 0;">
          <small>
            <b>Källa:</b>
            <a href="https://dataportalen.stockholm.se/dataportalen/GetMetaDataById?id=LvFeature15303196&showmetadataview"
               target="_blank">
               Stockholms stads öppna data
            </a>
          </small>"""
 
    popup_html = f"""
    <div style='font-family:Arial; font-size:13px; max-width:320px;'>
      <b>🚻 {name}</b><br>
      <b>Adress:</b> {address}<br>
      <b>Status:</b> {status}<br> 
      {avtal_html}
      <b>Beskrivning:</b> {description}<br>
      <b>Skapad:</b> {create_dt}<br>
      <b>Ändrad:</b> {change_dt}
      
      <b>Driftansvar:</b> {operator}<br>
      <b>Anläggningsnr:</b> {facility_id}<br>
      {wheelchair_html}
      📍 <a href='https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}'
            target='_blank'>{lat:.5f}, {lon:.5f}</a><br>

      {menu_html}
      {tools_html}
      {source_html}
      <!-- OSM_SECTION -->
    </div>
    """
    return popup_html


In [43]:
def build_external_media_links(tags: dict) -> str:
    links = []

    # Wikimedia Commons – image (direkt fil)
    image = tags.get("image")
    if image and image.startswith("https://commons.wikimedia.org/wiki/"):
        links.append(
            f'🖼️ <a href="{image}" target="_blank">Bild (Wikimedia Commons)</a>'
        )

    # Wikimedia Commons – category eller file via wikimedia_commons
    commons = tags.get("wikimedia_commons")
    if commons:
        commons_url = f"https://commons.wikimedia.org/wiki/{commons}"
        links.append(
            f'📂 <a href="{commons_url}" target="_blank">Commons-kategori</a>'
        )

    # Mapillary
    mapillary_key = tags.get("mapillary")
    if mapillary_key and mapillary_key.isdigit():
        mapillary_url = f"https://www.mapillary.com/app/?pKey={mapillary_key}"
        links.append(
            f'📷 <a href="{mapillary_url}" target="_blank">Mapillary-bild</a>'
        )

    # Panoramax – både panoramax och panoramax:*
    panoramax_ids = []

    if "panoramax" in tags:
        panoramax_ids.append(tags["panoramax"])

    for k, v in tags.items():
        if k.startswith("panoramax:") and v:
            panoramax_ids.append(v)

    for pano_id in panoramax_ids:
        pano_url = f"https://api.panoramax.xyz/#focus=pic&pic={pano_id}"
        links.append(
            f'🌐 <a href="{pano_url}" target="_blank">Panoramax-bild</a>'
        )

    if not links:
        return ""

    return (
        "<div style='margin-top:6px;'>"
        "<b>🔗 Media</b><br>"
        + "<br>".join(links) +
        "</div>"
    )


In [44]:
def build_osm_toilet_popup(
    osm_el: dict,
    *,
    lat: float,
    lon: float,
    lifecycle: str
) -> str:

    tags = osm_el.get("tags", {})
    osm_id = osm_el["id"]
    osm_type = osm_el["type"]

    row = {
        "lat": lat,
        "lon": lon,
        "Tillgänglighetsanpassad": tags.get("wheelchair", ""),
    }

    base_popup = build_toilet_popup(row, include_source=False)

    osm_object_url = f"https://www.openstreetmap.org/{osm_type}/{osm_id}"
    tags_html = render_osm_tags(tags)
    warnings_html = get_data_quality_warnings(tags)
    mapcomplete_link = build_mapcomplete_link(lat, lon, tags)

    media_links_html = build_external_media_links(tags)
    commons_thumb_html = build_commons_thumbnail(tags)
    if lifecycle == "disused":
        status_html = "🚫 Ur bruk (disused)"
    elif lifecycle == "active":
        status_html = "✅ I drift"
    else:
        status_html = "❓ Okänd status"

    status_block = f"""
    <b>Status:</b> {status_html}<br>
    """
    status_section = f"""
    <div style="margin-bottom:6px;">
      <b>🛠 Status & underhåll</b><br>
      Status: {status_html}<br>
    """

    if lifecycle == "disused":
        reopening = tags.get("expected_opening", "")
        note = tags.get("note", "")
        check_date = tags.get("check_date", "")
    
        if check_date:
            status_section += f"Senast kontrollerad: {check_date}<br>"
        if reopening:
            status_section += f"Planerad öppning: {reopening}<br>"
        if note:
            status_section += f"Kommentar: {note}<br>"
    
        status_section += "</div><hr style='margin:6px 0;'>"

    base_popup = status_section + base_popup

    osm_section = f"""
    <hr style="margin:6px 0;">
    <b>🧾 OpenStreetMap</b><br>
    <a href="{osm_object_url}" target="_blank">
      {osm_type.capitalize()} {osm_id}
    </a><br>
    {commons_thumb_html}
    {media_links_html}

    {warnings_html}

    <details style="margin-top:6px;">
      <summary style="cursor:pointer;font-weight:bold;">
        🏷️ OSM Tags
      </summary>
      {tags_html}
    </details>
    """
    return base_popup.replace("<!-- OSM_SECTION -->", osm_section)    


In [45]:
def build_mapcomplete_link(lat, lon, tags: dict) -> str:
    """
    Build MapComplete Toilets link with prefilled tags.
    """
    base = "https://mapcomplete.org/toilets.html"

    allowed_keys = {
        "wheelchair",
        "fee",
        "access",
        "unisex",
        "changing_table",
        "toilets:disposal",
        "toilets:position",
        "drinking_water",
    }

    preset_tags = {
        k: v for k, v in tags.items()
        if k in allowed_keys
    }

    tag_fragment = "&".join(
        f"{quote(k)}={quote(v)}" for k, v in preset_tags.items()
    )

    return (
        f"{base}?z=18&lat={lat}&lon={lon}"
        f"#preset=toilets"
        + (f"&{tag_fragment}" if tag_fragment else "")
    )


In [46]:
def get_data_quality_warnings(tags: dict, stale_years: int = 3) -> str:
    """
    Highlight questionable OSM data such as fixme or old check_date.
    """
    warnings = []

    # fixme=* is always suspicious
    if "fixme" in tags:
        warnings.append(
            f"⚠️ <b>fixme:</b> {tags.get('fixme')}"
        )

    # check_date staleness
    cd = tags.get("check_date")
    parsed = parse_osm_date(cd) if cd else None
    if parsed:
        age = date.today().year - parsed.year
        if age >= stale_years:
            warnings.append(
                f"⏰ <b>check_date:</b> {cd} (≈ {age} years old)"
            )

    if not warnings:
        return ""

    return (
        "<div style='background:#fff3cd;"
        "border:1px solid #ffecb5;"
        "padding:6px;border-radius:6px;"
        "margin-top:6px;font-size:12px;'>"
        "<b>Datakvalitet</b><br>"
        + "<br>".join(warnings) +
        "</div>"
    )


In [47]:
def render_osm_tags(tags: dict) -> str:
    """
    Render OSM tags as HTML with links to OSM Wiki for each key.
    """
    if not tags:
        return "<i>No tags</i>"

    rows = []
    for k, v in sorted(tags.items()):
        wiki = f"https://wiki.openstreetmap.org/wiki/Key:{quote(k)}"
        rows.append(
            f"<tr>"
            f"<td style='padding-right:6px;'><a href='{wiki}' target='_blank'><code>{k}</code></a></td>"
            f"<td><code>{v}</code></td>"
            f"</tr>"
        )

    return (
        "<table style='border-collapse:collapse;font-size:12px;'>"
        + "".join(rows)
        + "</table>"
    )


In [48]:
from datetime import date, datetime
def parse_osm_date(value: str):
    """
    Parse OSM-style date (YYYY-MM-DD). Return date or None.
    """
    try:
        return datetime.strptime(value, "%Y-%m-%d").date()
    except Exception:
        return None


### Open Street Map layer

In [49]:
import requests
import json
import os
import time
from pathlib import Path  

def get_lat_lon(el):
    if el["type"] == "node":
        return el.get("lat"), el.get("lon")
    if el["type"] in ("way", "relation"):
        center = el.get("center")
        if center:
            return center.get("lat"), center.get("lon")
    return None, None

# 🗺️ Overpass API
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# 🔍 Overpass query toilets Stockholms stad
# https://overpass-turbo.eu/s/2jOa
OVERPASS_QUERY = """
[out:json][timeout:60];
area(3600398021)->.searchArea;
nwr["amenity"="toilets"](area.searchArea);
out center;
"""
# 🔍 Overpass query toilets Stockholms stad and "disused:amenity": "toilets"
# 
OVERPASS_QUERY = """ 
[out:json][timeout:60];
area(3600398021)->.searchArea;
(
  nwr["amenity"="toilets"](area.searchArea);
  nwr["disused:amenity"="toilets"](area.searchArea);
);
out center;
"""


# 💾 Cache settings
CACHE_FILE = Path("osm_Sthlmkommun_cache.json")
CACHE_TTL = 60  # seconds (dev default)
#CACHE_TTL = 60 * 60 * 24 * 7  # 7 days (prod)

def is_cache_valid(cache_file: Path, ttl: int) -> bool:
    if not cache_file.exists():
        return False
    age = time.time() - cache_file.stat().st_mtime
    return age < ttl

def load_cache(cache_file: Path):
    with cache_file.open("r", encoding="utf-8") as f:
        return json.load(f)

def save_cache(cache_file: Path, data: dict):
    with cache_file.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def fetch_overpass(query: str) -> dict:
    response = requests.get(
        OVERPASS_URL,
        params={"data": query},
        timeout=90,
    )
    response.raise_for_status()
    return response.json()

def get_osm_toilets(
    *,
    use_cache: bool = True,
    force_refresh: bool = False,
) -> dict:
    """
    use_cache=True        → allow cache usage
    force_refresh=True    → bypass cache even if valid
    """

    if use_cache and not force_refresh and is_cache_valid(CACHE_FILE, CACHE_TTL):
        age = int(time.time() - CACHE_FILE.stat().st_mtime)
        print(f"📦 Using cache ({age}s old)")
        return load_cache(CACHE_FILE)

    print("🌍 Fetching toilets from Overpass API…")

    try:
        data = fetch_overpass(OVERPASS_QUERY)
        save_cache(CACHE_FILE, data)
        print(f"✅ Cached {len(data.get('elements', []))} objects")
        return data

    except Exception as e:
        print(f"❌ Overpass error: {e}")

        if CACHE_FILE.exists():
            print("⚠️ Falling back to stale cache")
            return load_cache(CACHE_FILE)

        raise  # no cache to fall back on

# 🚀 Example usage

# Normal dev run (cache enabled)
#osm_data = get_osm_toilets()

# Force refresh (bypass cache)  
osm_data = get_osm_toilets(force_refresh=True)

print(f"🐾 Found {len(osm_data.get('elements', []))} toilets")


🌍 Fetching toilets from Overpass API…
❌ Overpass error: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter?data=+%0A%5Bout%3Ajson%5D%5Btimeout%3A60%5D%3B%0Aarea%283600398021%29-%3E.searchArea%3B%0A%28%0A++nwr%5B%22amenity%22%3D%22toilets%22%5D%28area.searchArea%29%3B%0A++nwr%5B%22disused%3Aamenity%22%3D%22toilets%22%5D%28area.searchArea%29%3B%0A%29%3B%0Aout+center%3B%0A
⚠️ Falling back to stale cache
🐾 Found 253 toilets


In [50]:
from datetime import datetime, UTC
from pathlib import Path
import json

def save_snapshot(data: dict, prefix: str = "osm_toilets_history"):
    # timezone-aware UTC datetime
    date_str = datetime.now(UTC).strftime("%Y_%m_%d")  # YYMMDD
    filename = f"{prefix}_{date_str}.json"
    path = Path(filename)

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"🗂️ Snapshot saved to {path.resolve()}")
    return path


In [51]:
save_snapshot(osm_data)

🗂️ Snapshot saved to /Users/salgo/Documents/GitHub/Offentliga-toaletter/Notebook/osm_toilets_history_2026_05_05.json


PosixPath('osm_toilets_history_2026_05_05.json')

In [52]:
#osm_data

In [53]:
with open("osm_data.json", "w", encoding="utf-8") as f:
    json.dump(osm_data, f, indent=2, ensure_ascii=False)

# Generate Map

In [54]:
import folium 
m = folium.Map(location=[59.3293, 18.0686], zoom_start=11)
date_str = datetime.now(UTC).strftime("%Y_%m_%d")  # YYMMDD
filenamecsv = f"Sthlm_{date_str}.csv"
gdf.to_csv(filenamecsv)
print(filenamecsv, "skapad")

Sthlm_2026_05_05.csv skapad


In [55]:
from shapely.geometry import base

# Ta bort alla geometrier utom den huvudsakliga
gdf_clean = gdf.copy()

# ta bort "geometry_centroid" och andra geometrikolumner
geom_cols = [c for c in gdf_clean.columns 
             if c != "geometry" and gdf_clean[c].apply(lambda x: isinstance(x, base.BaseGeometry)).any()]
if geom_cols:
    print("🧹 Tar bort extra geometri-kolumner:", geom_cols)
    gdf_clean = gdf_clean.drop(columns=geom_cols, errors="ignore")

# Skapa Folium-lager utan att försöka serialisera själva geometry-kolumnen
stad_layer = folium.FeatureGroup(name="🟠 Toaletter (Stockholms stad)")

for _, row in gdf_clean.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8,
        fill=True,
        fill_color="#ff6600",
        color="#cc5500",
        weight=2,
        fill_opacity=0.7,
        popup=folium.Popup(build_toilet_popup(row), max_width=350),
    ).add_to(stad_layer)


stad_layer.add_to(m)

gdf_danfo = gdf_clean[
    gdf_clean["Huvudman"]
    .astype(str)
    .str.strip()
    .eq("Danfo")
].copy()

Danfo_layer = folium.FeatureGroup(
    name="🟣 Endast Stockholm stad Danfo (nytt avtal)",
    show=False
)

for _, row in gdf_danfo.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8,                 # större → analytiskt fokus
        fill=True,
        fill_color="#7b2cff",
        color="#4b1a99",
        weight=2,
        fill_opacity=0.85,
        popup=folium.Popup(build_toilet_popup(row), max_width=350),
    ).add_to(Danfo_layer)

Danfo_layer.add_to(m)


🧹 Tar bort extra geometri-kolumner: ['geometry_centroid']


In [56]:
gdf_danfo

,id,Placering,Huvudman,Drift,Typ,Modell,Stadsdelsnämnder,Anläggningsnummer,Cykelpump,Dricksfontän,geometry,lon,lat
26,129,Hamngatan 2 - Nybroplan,Danfo,Danfo,Toalett,Dubbel,Norra innerstaden,G055,Ja,Ja,POINT (154284.977 6579752.763),18.075281,59.332872
27,131,Akallastigen 6 - Akalla By,Danfo,Danfo,Toalett,Enkel,Järva,G099,Nej,Nej,POINT (144969.689 6588700.874),17.911415,59.413187
28,133,Husbystigen 9 - Husby Gård,Danfo,Danfo,Toalett,Enkel,Järva,G098,Nej,Nej,POINT (145382.018 6588037.657),17.918691,59.407238
29,136,Mälartorget 19,Danfo,Danfo,Toalett,Dubbel,Södermalm,G049,Nej,Ja,POINT (153934.062 6578623.459),18.069096,59.322738
30,137,Slottsbacken/Österlånggatan,Danfo,Danfo,Toalett,Trippel,Södermalm,G053,Ja,Nej,POINT (154159.639 6579000.892),18.073065,59.326124
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,198,Vittangiplan 28 - Vällingby Torg,Danfo,Danfo,Toalett,Enkel,Hässelby-Vällingby,G080,Nej,Nej,POINT (142729.311 6583029.987),17.872153,59.362249
87,199,Andholmsvägen 5 - Vårbergstoppen,Danfo,Danfo,Toalett,Enkel,Skärholmen,G094,Nej,Nej,POINT (143192.258 6573016.898),17.880609,59.272373
88,200,Wollmar Yxkullsgatan,Danfo,Danfo,Toalett,Enkel,Södermalm,G056,Nej,Nej,POINT (153218.805 6577880.037),18.056522,59.316071
89,201,Österholmsstigen - Västerholmsparken,Danfo,Danfo,Toalett,Enkel,Skärholmen,G039,Nej,Nej,POINT (144231.827 6572922.827),17.898843,59.271544


In [57]:
from datetime import datetime, timezone

generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


### Add OSM to map layer OSM

2026-01-23: 'node': 181, 'way': 56}

In [58]:
# should be 237 
rendered = []

for el in osm_data["elements"]:
    lat, lon = get_lat_lon(el)
    if lat is None:
        continue
    rendered.append(el)

len(rendered)


253

In [59]:
#assert len(rendered) == 240, f"Expected 240 toilets, rendered {len(rendered)}"

In [60]:
from datetime import date, datetime

STALE_YEARS = 3

def parse_osm_date(value):
    try:
        return datetime.strptime(value, "%Y-%m-%d").date()
    except Exception:
        return None


def classify_toilet(tags: dict) -> str:
    """
    Returns one of: fixme, stale, unknown, ok
    """
    if "fixme" in tags:
        return "fixme"

    cd = tags.get("check_date")
    if not cd:
        return "unknown"

    parsed = parse_osm_date(cd)
    if not parsed:
        return "unknown"

    age_years = date.today().year - parsed.year
    if age_years >= STALE_YEARS:
        return "stale"

    return "ok"


In [61]:
def classify_toilet_lifecycle(tags: dict) -> str:
    if tags.get("disused:amenity") == "toilets":
        return "disused"
    if tags.get("amenity") == "toilets":
        return "active"
    return "other"

In [62]:
def marker_color(tags: dict) -> str:
    if tags.get("disused:amenity") == "toilets":
        return "red"
    if tags.get("amenity") == "toilets":
        return "green"
    return "gray"
    

In [63]:
import folium

elements = osm_data.get("elements", [])

fg_active = folium.FeatureGroup(
    name="🚻 Open Street Map Toaletter – aktiva",
    show=True
)

fg_disused = folium.FeatureGroup(
    name="🚫 Open Street Map Toaletter – avstängda",
    show=True
)

for el in osm_data["elements"]:
    lat, lon = get_lat_lon(el)
    if lat is None or lon is None:
        continue

    tags = el.get("tags", {})

    if lat is None or lon is None:
        continue
    lifecycle = classify_toilet_lifecycle(tags)
    colorMarker = marker_color(tags)
    popup_html = build_osm_toilet_popup(el, lat=lat, lon=lon,
                    lifecycle=lifecycle)
    
    marker = folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=380),
        icon=folium.Icon(
            icon="restroom",
            prefix="fa",
            color=colorMarker,
        ),
    )
    if lifecycle == "active":
        fg_active.add_child(marker)
    elif lifecycle == "disused":
        fg_disused.add_child(marker)

fg_active.add_to(m)
fg_disused.add_to(m)




### Add legend and Sthlm

In [64]:
legend_html = f"""
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 320px;
    z-index: 10000;
    font-size: 12px;
">

  <!-- Header / Toggle -->
  <div onclick="var c=document.getElementById('legend-content');
                c.style.display = (c.style.display === 'none') ? 'block' : 'none';"
       style="
        cursor: pointer;
        background: white;
        padding: 8px;
        border: 2px solid #ccc;
        border-radius: 6px 6px 0 0;
        font-weight: bold;
       ">
    🟠 Offentliga toaletter – Stockholms stad v{Mapversion}
    <span style="float:right;">▼</span>
  </div>

  <!-- Collapsible content -->
  <div id="legend-content" style="
        display: none;
        background: white;
        padding: 10px;
        border: 2px solid #ccc;
        border-top: none;
        border-radius: 0 0 6px 6px;
  ">

    <small><i>Karta genererad: {generated_at}</i></small><br><br>

    <b>Syfte</b><br>
    Datat används för att verifiera och förbättra OSM.  
    Stockholms stad gör om en hel del – se issue
    (<i>amenity=toilets</i>).<br><br>

    <b>Mer info</b><br>
    • <a href="https://github.com/salgo60/Offentliga-toaletter/issues/19" target="_blank">
      GitHub Issue #19
    </a><br>
    • <a href="https://dataportalen.stockholm.se/dataportalen/GetMetaDataById?id=LvFeature15303196&showmetadataview" target="_blank">
      Stockholms stads öppna data
    </a>

  </div>
</div>
"""


In [65]:
#legend_html

In [66]:
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

In [67]:
from branca.element import MacroElement
from jinja2 import Template

class MinPlatsControl(MacroElement):
    _template = Template("""
        {% macro script(this, kwargs) %}
        var minPlats = L.control({position: 'bottomright'});

        minPlats.onAdd = function (map) {
            var div = L.DomUtil.create('div', 'min-plats-control');
            div.innerHTML = '📍 <strong>Min plats</strong>';

            // Gör klickbar
            L.DomEvent.disableClickPropagation(div);
            L.DomEvent.on(div, 'click', function (e) {
                map.locate({setView: true, maxZoom: 16});
            });

            return div;
        };

        minPlats.addTo({{this._parent.get_name()}});
        {% endmacro %}
    """)

m.add_child(MinPlatsControl())
from folium import Element

css = """
<style>
.min-plats-control {
    cursor: pointer;
    background: #cfe8ef;
    padding: 8px 12px;
    border-radius: 6px;
    box-shadow: 0 0 6px rgba(0,0,0,0.3);
    font-size: 16px;
    user-select: none;
}
.min-plats-control:hover {
    background: #b9dde7;
}
</style>
"""
m.get_root().html.add_child(Element(css))


In [68]:
m

In [69]:
# Exportera kartan https://salgo60.github.io/Offentliga-toaletter/Notebook/Toaletter_sthlm.html
html_map_file = "Toaletter_sthml_" + Mapversion.replace(".", "_") + ".html"
m.save(html_map_file) 
m.save("Toaletter_sthlm_latest.html") 

In [70]:
html_map_file

'Toaletter_sthml_0_9.html'

In [71]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Total time elapsed: {:.2f} seconds".format(elapsed_time))

Date: 2026-05-05 11:49:12
Total time elapsed: 2.25 seconds
